# Capstone — Refresh / Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring

This notebook is the final capstone analysis for the FlyRank ML internship. It turns the earlier weekly work into one reproducible end-to-end study: define the decision, prepare a leakage-safe dataset, build a transparent baseline, train learned models, validate them with a client-held-out split, compare results on the same test population, and produce a ranked refresh/review queue for human decision support.

**Public-safe framing:** the outcome is an observed decline label in the available dataset. The model does not prove causation, predict Google's algorithm, or guarantee that a refresh will improve performance.

## 1. Question

### Research question
**Which content pages should be reviewed first for refresh, expansion, protection, or monitoring based on observed search-performance and content signals?**

### Decision supported
The output is a ranked review queue. An SEO specialist, content manager, or analyst can use the ranking to decide which pages deserve human review first. The cost of a wrong call is mainly wasted editorial/analysis effort or missed review of a page that later declines, so the evaluation emphasizes **Precision@K** for a small review queue.

### Target
A page is labeled **declining** when `trend_direction == "down"`. This is an observed outcome/proxy for refresh opportunity, not a label saying that a refresh is definitely required or will succeed.

In [38]:
!ls -la /content

total 1948
drwxr-xr-x 1 root root    4096 Sep  2 18:54 .
drwxr-xr-x 1 root root    4096 Sep  2 18:40 ..
drwxr-xr-x 4 root root    4096 Aug 24 13:21 .config
drwxr-xr-x 4 root root    4096 Sep  2 18:55 flyrank
-rw-r--r-- 1 root root 1973659 Sep  2 18:54 flyrank-repo.zip
drwxr-xr-x 1 root root    4096 Aug 24 13:21 sample_data


In [39]:
!wget -O /content/flyrank-repo.zip \
"https://github.com/ShahvezAli784/-flyrank-machine-learning/archive/refs/heads/main.zip"

--2026-09-02 19:04:00--  https://github.com/ShahvezAli784/-flyrank-machine-learning/archive/refs/heads/main.zip
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/ShahvezAli784/-flyrank-machine-learning/zip/refs/heads/main [following]
--2026-09-02 19:04:00--  https://codeload.github.com/ShahvezAli784/-flyrank-machine-learning/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 140.82.121.9
Connecting to codeload.github.com (codeload.github.com)|140.82.121.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘/content/flyrank-repo.zip’

/content/flyrank-re     [  <=>               ]   1.88M  4.35MB/s    in 0.4s    

2026-09-02 19:04:01 (4.35 MB/s) - ‘/content/flyrank-repo.zip’ saved [1973659]



In [40]:
import zipfile
from pathlib import Path

zip_path = Path("/content/flyrank-repo.zip")
extract_path = Path("/content/flyrank")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Repository extracted.")

Repository extracted.


In [41]:
!find /content/flyrank -name "content_refresh_anonymized.csv"

/content/flyrank/-flyrank-machine-learning-main/data/raw/content_refresh_anonymized.csv


In [42]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

RANDOM_STATE = 42

# Repository root in Google Colab
REPO_ROOT = Path("/content/flyrank/-flyrank-machine-learning-main")

# Dataset
DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"

# Output directory
OUTPUT_DIR = REPO_ROOT / "work/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Repository root:", REPO_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
print("Output directory:", OUTPUT_DIR)


Repository root: /content/flyrank/-flyrank-machine-learning-main
Dataset path: /content/flyrank/-flyrank-machine-learning-main/data/raw/content_refresh_anonymized.csv
Dataset shape: (30000, 44)
Output directory: /content/flyrank/-flyrank-machine-learning-main/work/outputs


## 2. Data

This capstone uses the repository's **30,000-row anonymized starter release** (`content_refresh_anonymized.csv`). Each row represents one content item with search, traffic, engagement, position, content, and freshness fields plus the observed trend outcome.

For the modeling population, I keep pages with `impressions_90d > 0` and `content_age_days >= 90`, then keep one row per `content_id`. These filters match the population used in the earlier baseline/model work and avoid treating brand-new or completely unseen pages as ordinary refresh candidates.

The following fields are deliberately excluded from the feature matrix:
- `content_id`, `client_id`: identifiers; `client_id` is used only for grouped validation.
- `trend_direction`, `trend_pct`: outcome/label-derived fields.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` and the corresponding `*_prev_30d` fields: these windows directly determine the observed trend and would leak the outcome.
- `provider_used`, `model_used`: not needed for this decision-support model.

No client names, domains, URLs, private queries, or credentials are used in this notebook.

In [43]:
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

model_df = (
    df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
    .drop_duplicates(subset=["content_id"])
    .reset_index(drop=True)
)

print(f"Modeling rows: {len(model_df):,}")
print(f"Clients represented: {model_df['client_id'].nunique():,}")
print(f"Declining-label rate: {model_df['is_declining_label'].mean():.3%}")
print("\nTrend labels:")
print(model_df["trend_direction"].value_counts())

Modeling rows: 30,000
Clients represented: 32
Declining-label rate: 54.207%

Trend labels:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Methodology

### Feature contract
The model uses decision-time content, keyword, traffic, engagement, position, and freshness signals. Numeric missing values are median-imputed with missingness indicators; categorical missing values use the most frequent training category; unseen categories are ignored safely by the encoder.

### Validation design
I use a deterministic **80/20 client-held-out split** with `random_state=42`. Entire clients are held out from the test set rather than randomly mixing rows from the same client. This is a stronger test of whether the learned patterns generalize beyond client-specific repetition. The same held-out rows are used for both the Week-4 baseline and all learned models.

### Models
- **Week-4 baseline:** transparent rule using staleness + visibility/freshness percentiles.
- **Logistic Regression:** simple interpretable learned classifier.
- **Decision Tree:** tests nonlinear splits with constrained depth.
- **Random Forest:** tests whether ensembles and feature interactions improve the ranking.

### Model selection
The primary decision metric is **Precision@50** because the intended use is a small ranked review queue. PR-AUC and ROC-AUC provide additional discrimination context.

In [44]:
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "age_tier_order", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier"
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "is_declining_label"

LEAKAGE_COLUMNS = [
    "content_id", "client_id", "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used"
]

assert not set(LEAKAGE_COLUMNS).intersection(FEATURES)
print(f"Numeric features: {len(NUMERIC_FEATURES)}")
print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Total model inputs: {len(FEATURES)}")

Numeric features: 23
Categorical features: 9
Total model inputs: 32


In [45]:
client_series = model_df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_indices = np.flatnonzero(~test_mask)
test_indices = np.flatnonzero(test_mask)

y = model_df[TARGET].astype(int)
if y.iloc[train_indices].nunique() != 2 or y.iloc[test_indices].nunique() != 2:
    raise ValueError("Client-held-out split must contain both target classes in train and test.")

X = model_df[FEATURES].copy()
y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

print(f"Train rows: {len(train_indices):,}")
print(f"Test rows: {len(test_indices):,}")
print(f"Train clients: {len(set(client_series.iloc[train_indices])):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Train declining rate: {y_train.mean():.3%}")
print(f"Test declining rate: {y_test.mean():.3%}")

Train rows: 27,675
Test rows: 2,325
Train clients: 26
Test clients: 6
Train declining rate: 55.476%
Test declining rate: 39.097%


### Leakage audit
The target is derived from `trend_direction`. The model therefore never receives that field, `trend_pct`, or the recent/previous 30-day windows that directly define the trend calculation. IDs are also excluded from the feature matrix. `client_id` is retained only to construct the grouped split.

## 4. Results (vs baseline)

The baseline and learned models are evaluated on the **same client-held-out test set**. This is the central comparison required for the capstone: a more complex model only earns its place if it improves the decision-relevant ranking.

In [46]:
def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

# Week-4 transparent baseline, rebuilt only on the held-out test population.
test_frame = model_df.iloc[test_indices].copy()
test_frame["visibility_percentile"] = np.log1p(test_frame["impressions_90d"]).rank(
    method="average", pct=True
)
test_frame["freshness_risk_percentile"] = test_frame["days_since_last_update"].rank(
    method="average", pct=True
)
test_frame["stale_visible"] = (
    (test_frame["days_since_last_update"] >= 180)
    & (test_frame["impressions_90d"] >= 500)
).astype(int)
test_frame["baseline_score"] = (
    2.0 * test_frame["stale_visible"]
    + 0.5 * test_frame["visibility_percentile"]
    + 0.5 * test_frame["freshness_risk_percentile"]
)
baseline_scores = test_frame["baseline_score"].to_numpy()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
], remainder="drop")

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample", n_estimators=200, max_depth=10,
        min_samples_leaf=25, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

results = [{
    "Model": "Week-4 Baseline",
    "Precision@10": precision_at_k(y_test, baseline_scores, 10),
    "Precision@20": precision_at_k(y_test, baseline_scores, 20),
    "Precision@50": precision_at_k(y_test, baseline_scores, 50),
    "ROC-AUC": np.nan, "PR-AUC": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan,
}]
trained_models, probabilities = {}, {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X.iloc[train_indices], y.iloc[train_indices])
    prob = pipe.predict_proba(X.iloc[test_indices])[:, 1]
    pred = (prob >= 0.5).astype(int)
    trained_models[name] = pipe
    probabilities[name] = prob
    results.append({
        "Model": name,
        "Precision@10": precision_at_k(y_test, prob, 10),
        "Precision@20": precision_at_k(y_test, prob, 20),
        "Precision@50": precision_at_k(y_test, prob, 50),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    })

comparison = pd.DataFrame(results).sort_values(
    ["Precision@50", "PR-AUC"], ascending=False, na_position="last"
).reset_index(drop=True)

display(comparison.round(3))
comparison.to_csv(OUTPUT_DIR / "capstone_model_comparison.csv", index=False)
print("Saved:", OUTPUT_DIR / "capstone_model_comparison.csv")

,Model,Precision@10,Precision@20,Precision@50,ROC-AUC,PR-AUC,Precision,Recall,F1
0,Logistic Regression,0.8,0.80,0.82,0.731,0.626,0.647,0.570,0.606
1,Random Forest,0.8,0.70,0.72,0.746,0.607,0.566,0.741,0.642
2,Decision Tree,0.6,0.45,0.58,0.742,0.575,0.569,0.716,0.634
3,Week-4 Baseline,0.7,0.35,0.26,NaN,NaN,NaN,NaN,NaN


Saved: /content/flyrank/-flyrank-machine-learning-main/work/outputs/capstone_model_comparison.csv


In [47]:
learned = comparison[comparison["Model"] != "Week-4 Baseline"].copy()
best_row = learned.sort_values(["Precision@50", "PR-AUC", "ROC-AUC"], ascending=False).iloc[0]
best_model_name = best_row["Model"]

print("Selected learned model:", best_model_name)
print(f"Precision@50: {best_row['Precision@50']:.3f}")
print(f"PR-AUC: {best_row['PR-AUC']:.3f}")
print(f"ROC-AUC: {best_row['ROC-AUC']:.3f}")

Selected learned model: Logistic Regression
Precision@50: 0.820
PR-AUC: 0.626
ROC-AUC: 0.731


### Interpretation
The selected model is chosen by held-out Precision@50 rather than by complexity. Feature importance and error analysis are used as review aids, not as causal evidence. A strong feature association means the model used that signal for discrimination; it does not mean changing the signal will cause rankings or traffic to improve.

In [48]:
best_prob = probabilities[best_model_name]
error_frame = model_df.iloc[test_indices][
    ["content_id", "is_declining_label", "impressions_90d", "days_since_last_update",
     "avg_position", "ctr", "content_age_days"]
].copy()
error_frame["predicted_probability"] = best_prob
error_frame["predicted_label"] = (best_prob >= 0.5).astype(int)
error_frame["error_type"] = np.select(
    [
        (error_frame["predicted_label"] == 1) & (error_frame["is_declining_label"] == 0),
        (error_frame["predicted_label"] == 0) & (error_frame["is_declining_label"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

print("False positives — examples for human review:")
display(error_frame[error_frame.error_type == "false_positive"].sort_values("predicted_probability", ascending=False).head(3).round(3))
print("False negatives — examples for human review:")
display(error_frame[error_frame.error_type == "false_negative"].sort_values("predicted_probability").head(3).round(3))

cm = confusion_matrix(y_test, (best_prob >= 0.5).astype(int))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)

False positives — examples for human review:


,content_id,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,predicted_probability,predicted_label,error_type
3982,content_9b4ddfa91f64,0,121,20,8.5,0.00,133,0.790,1,false_positive
29266,content_423c7cf07765,0,1356,20,10.5,0.15,104,0.753,1,false_positive
27764,content_9284688e3982,0,674,20,7.4,0.15,104,0.743,1,false_positive


False negatives — examples for human review:


,content_id,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,predicted_probability,predicted_label,error_type
17188,content_a0a76e94ade5,1,19,20,9.2,0.0,175,0.0,0,false_negative
25755,content_a8cee66e4788,1,1,20,2.0,100.0,489,0.0,0,false_negative
1068,content_836b4163cf30,1,95,8,28.2,0.0,140,0.0,0,false_negative


Confusion matrix [[TN, FP], [FN, TP]]:
[[1133  283]
 [ 391  518]]


In [49]:
best_pipeline = trained_models[best_model_name]
pre = best_pipeline.named_steps["preprocessor"]
estimator = best_pipeline.named_steps["model"]
feature_names = pre.get_feature_names_out()

if hasattr(estimator, "feature_importances_"):
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": estimator.feature_importances_
    }).sort_values("importance", ascending=False)
elif hasattr(estimator, "coef_"):
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": np.abs(estimator.coef_[0])
    }).sort_values("importance", ascending=False)
else:
    importance = pd.DataFrame(columns=["feature", "importance"])

print(f"Top 10 model signals — {best_model_name}:")
display(importance.head(10).round(4))
importance.head(20).to_csv(OUTPUT_DIR / "capstone_top_features.csv", index=False)

Top 10 model signals — Logistic Regression:


,feature,importance
3,numeric__word_count,1.0280
63,categorical__position_tier_top_3,0.9115
4,numeric__char_count,0.8689
9,numeric__users_90d,0.7581
8,numeric__sessions_90d,0.6838
13,numeric__days_with_impressions,0.5897
14,numeric__days_with_sessions,0.4625
50,categorical__word_count_tier_<1000,0.4554
49,categorical__word_count_tier_3500+,0.4083
37,categorical__main_intent_navigational,0.4079


## 5. Limitations and honest framing

1. **Observed-label limitation:** `is_declining_label` represents an observed trend state, not whether a refresh would actually create a positive outcome.
2. **Observational data:** the model can identify associations and rank review candidates, but it cannot establish that a particular content change causes traffic, visibility, or rankings to improve.
3. **Dataset scope:** this analysis uses the anonymized starter release available in the repository, so findings should not be generalized automatically to every site or future period.
4. **Grouped validation is not temporal validation:** holding out clients reduces client-specific leakage, but it does not prove future-time performance.
5. **Human review remains necessary:** a high score is a prioritization signal. Editors should inspect search intent, content quality, business context, duplication/cannibalization, and recent changes before acting.
6. **No Google-algorithm claim:** these results do not identify or prove ranking factors used by Google.

In [ ]:
# Fit the selected model on all modeling rows to create the final decision-support ranking.
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[best_model_name]),
])
final_pipeline.fit(X, y)
model_df["decline_probability"] = final_pipeline.predict_proba(X)[:, 1]

ranked = model_df.copy()
ranked["stale_flag"] = (ranked["days_since_last_update"] >= 180)
raked_visible = ranked["impressions_90d"] >= 500
ranked["visible_flag"] = raked_visible

def reason_codes(row):
    reasons = []
    if row["stale_flag"] and row["visible_flag"]:
        reasons.append("stale + visible")
    elif row["stale_flag"]:
        reasons.append("stale content")
    if row["ctr"] < ranked["ctr"].median() and row["impressions_90d"] >= 500:
        reasons.append("visible CTR review")
    if row["engagement_rate"] < ranked["engagement_rate"].median() and row["sessions_90d"] > 0:
        reasons.append("engagement review")
    if row["content_age_days"] >= ranked["content_age_days"].quantile(0.75):
        reasons.append("older content")
    return "; ".join(reasons) if reasons else "model-ranked review"

ranked["reason_code"] = ranked.apply(reason_codes, axis=1)
ranked["recommended_action"] = np.select(
    [
        (ranked["decline_probability"] >= 0.75) & ranked["stale_flag"] & ranked["visible_flag"],
        ranked["decline_probability"] >= 0.75,
        ranked["decline_probability"] >= 0.50,
    ],
    ["refresh_and_review", "priority_review", "monitor_closely"],
    default="monitor",
)
ranked = ranked.sort_values("decline_probability", ascending=False).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked) + 1)

queue_cols = ["rank", "decline_probability", "recommended_action", "reason_code"]
queue = ranked[queue_cols].copy()
queue.head(20).style.format({"decline_probability": "{:.3f}"})

## 6. Ranked recommendations

The final output is a **review queue**, not an automatic publishing or editing system. The model probability ranks pages by estimated likelihood of the observed declining label. Reason codes add a human-readable explanation using safe, decision-time context.

In [36]:
# Fit the selected model on all modeling rows to create the final decision-support ranking.
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[best_model_name]),
])
final_pipeline.fit(X, y)
model_df["decline_probability"] = final_pipeline.predict_proba(X)[:, 1]

ranked = model_df.copy()
ranked["stale_flag"] = (ranked["days_since_last_update"] >= 180)
raked_visible = ranked["impressions_90d"] >= 500
ranked["visible_flag"] = raked_visible

def reason_codes(row):
    reasons = []
    if row["stale_flag"] and row["visible_flag"]:
        reasons.append("stale + visible")
    elif row["stale_flag"]:
        reasons.append("stale content")
    if row["ctr"] < ranked["ctr"].median() and row["impressions_90d"] >= 500:
        reasons.append("visible CTR review")
    if row["engagement_rate"] < ranked["engagement_rate"].median() and row["sessions_90d"] > 0:
        reasons.append("engagement review")
    if row["content_age_days"] >= ranked["content_age_days"].quantile(0.75):
        reasons.append("older content")
    return "; ".join(reasons) if reasons else "model-ranked review"

ranked["reason_code"] = ranked.apply(reason_codes, axis=1)
ranked["recommended_action"] = np.select(
    [
        (ranked["decline_probability"] >= 0.75) & ranked["stale_flag"] & ranked["visible_flag"],
        ranked["decline_probability"] >= 0.75,
        ranked["decline_probability"] >= 0.50,
    ],
    ["refresh_and_review", "priority_review", "monitor_closely"],
    default="monitor",
)
ranked = ranked.sort_values("decline_probability", ascending=False).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked) + 1)

queue_cols = ["rank", "decline_probability", "recommended_action", "reason_code"]
queue = ranked[queue_cols].copy()
queue.head(20).style.format({"decline_probability": "{:.3f}"})

,rank,decline_probability,recommended_action,reason_code
0,1,1.000,priority_review,older content
1,2,1.000,priority_review,model-ranked review
2,3,0.999,priority_review,model-ranked review
3,4,0.996,priority_review,model-ranked review
4,5,0.984,priority_review,model-ranked review
5,6,0.980,priority_review,model-ranked review
6,7,0.961,priority_review,model-ranked review
7,8,0.960,priority_review,model-ranked review
8,9,0.959,priority_review,model-ranked review
9,10,0.953,priority_review,model-ranked review


In [37]:
# Public-safe queue export: no content/client IDs, URLs, or private query fields.
public_queue = queue.head(50).copy()
public_queue.to_csv(OUTPUT_DIR / "capstone_ranked_recommendations_top50.csv", index=False)

action_mix = ranked["recommended_action"].value_counts().rename_axis("action").reset_index(name="rows")
reason_mix = ranked["reason_code"].value_counts().head(10).rename_axis("reason_code").reset_index(name="rows")

print("Top-50 action mix:")
print(public_queue["recommended_action"].value_counts())
print("\nFull-queue action mix:")
display(action_mix)
print("\nTop reason codes:")
display(reason_mix)

action_mix.to_csv(OUTPUT_DIR / "capstone_action_mix.csv", index=False)
reason_mix.to_csv(OUTPUT_DIR / "capstone_reason_mix.csv", index=False)

Top-50 action mix:
recommended_action
priority_review    50
Name: count, dtype: int64

Full-queue action mix:


,action,rows
0,monitor_closely,14518
1,monitor,13699
2,priority_review,1783



Top reason codes:


,reason_code,rows
0,model-ranked review,19293
1,older content,6400
2,visible CTR review,2777
3,visible CTR review; older content,1356
4,stale content,149
5,stale + visible,14
6,stale content; older content,8
7,stale + visible; visible CTR review,3


## 7. Artifacts for the research paper

The next cells create the minimum charts/tables needed by the deployed paper: model-vs-baseline ranking performance, the action mix, and the strongest model signals. The paper should describe these as measured/observed results and decision-support evidence.

In [ ]:
# Chart 1 — Precision@K comparison
metric_cols = ["Precision@10", "Precision@20", "Precision@50"]
plot_df = comparison.set_index("Model")[metric_cols]
ax = plot_df.T.plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Precision")
ax.set_xlabel("Review queue size")
ax.set_title("Model vs baseline ranking precision")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "capstone_precision_at_k.png", dpi=160)
plt.show()

In [ ]:
# Chart 2 — Action mix
action_counts = ranked["recommended_action"].value_counts().sort_values(ascending=False)
ax = action_counts.plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Number of content items")
ax.set_xlabel("Recommended action")
ax.set_title("Final decision-support action mix")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "capstone_action_mix.png", dpi=160)
plt.show()

In [ ]:
# Chart 3 — Top model signals
top_imp = importance.head(10).sort_values("importance")
ax = top_imp.set_index("feature")["importance"].plot(kind="barh", figsize=(9, 6))
ax.set_xlabel("Model importance")
ax.set_ylabel("Feature")
ax.set_title(f"Top model signals — {best_model_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "capstone_top_feature_importance.png", dpi=160)
plt.show()

## 8. Reproducibility

Run this notebook from a fresh clone of the repository after installing the packages in `requirements.txt`. The analysis uses `random_state=42` for the grouped split and all stochastic classifiers. Generated paper-ready artifacts are written to `work/outputs/`.

Key outputs:
- `capstone_model_comparison.csv`
- `capstone_top_features.csv`
- `capstone_ranked_recommendations_top50.csv`
- `capstone_action_mix.csv`
- `capstone_reason_mix.csv`
- `capstone_precision_at_k.png`
- `capstone_action_mix.png`
- `capstone_top_feature_importance.png`

## Final self-check

- [x] Research question and decision are explicit.
- [x] Data population and exclusions are documented.
- [x] Label definition is explicit.
- [x] Leakage-prone fields are excluded.
- [x] Baseline and learned models use the same held-out test population.
- [x] Client-held-out validation is deterministic.
- [x] Precision@10/@20/@50, ROC-AUC, PR-AUC, precision, recall, and F1 are reported.
- [x] Error examples and model signals are inspected.
- [x] Ranked recommendations include reason codes.
- [x] Public-safe artifacts avoid client names, URLs, and private queries.
- [x] Claims are framed as observed, measured, directional, and decision-support.
- [ ] Run **Runtime → Run all** from a clean environment and commit the executed notebook.